segment_inegi_agland.py -- James Sayre, sayrejay@pm.me

Intersect ADCs with INEGI agmask information.

In [ ]:
### Programs
import os, sys
os.environ['USE_PYGEOS'] = '0'
import regionmask
import geopandas as gpd
import shapely
import xarray as xr
from shapely.geometry import Polygon
from shapely.geometry import Point
from shapely.ops import cascaded_union

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

### Directories
topdir             =  "/home/j/Dropbox/Projects/"
projectdir         =  os.path.join(topdir,    "Maize_prediction")
cropdir            =  os.path.join(topdir,    "Crop_misallocation")
sciagadir          =  os.path.join(cropdir,   "data", "SCIAGA")
datadir            =  os.path.join(projectdir,"Data")
amcadir            =  os.path.join(datadir,   "INEGI","Areas_Censal_Agropecuario_2016")
agland_dir         =  os.path.join(datadir,   "Mexico_agland", "Inputs")
agland_out_dir     =  os.path.join(datadir,   "Mexico_agland", "Outputs")


### Inputs
adcshp             =  os.path.join(os.path.expanduser("~"), "Dropbox", "Projects", "Maize_prediction", "Data", "Shapefiles", "adc_shapefile.shp")
ca2007ageb         =  os.path.join(sciagadir,  "CA2007_ageb_poly.shp")
ca2016adc          =  os.path.join(amcadir,    "census_areas.shp")
agland_path        =  os.path.join(agland_dir, "agland_udsIV.shp")
ca2007mun          =  os.path.join(sciagadir,  "CA2007_mun_fromadc_poly.shp")
ca2016mun          =  os.path.join(amcadir,    "muns_from_ADC16_areas.shp")

### Outputs
agland_07_adc      =  os.path.join(agland_out_dir, "inegi_agland_07_adc.shp")
agland_16_adc      =  os.path.join(agland_out_dir, "inegi_agland_16_adc.shp")
agland_07_mun      =  os.path.join(agland_out_dir, "inegi_agland_07_mun.shp")
agland_16_mun      =  os.path.join(agland_out_dir, "inegi_agland_16_mun.shp")

In [13]:
adc07_df = gpd.read_file(adcshp)
adc07_df['ha'] = adc07_df.to_crs(epsg=6372).geometry.area/10000.0

In [21]:
### Compute size of agebs
ageb_df       = gpd.read_file(ca2007ageb)
ageb_df['ha'] = ageb_df.to_crs(epsg=6372).geometry.area/10000.0

In [34]:
### Municipality shapefiles
mun07_df = gpd.read_file(ca2007mun)
mun16_df = gpd.read_file(ca2016mun)[['muncode','geometry']]
mun16_df = mun16_df.set_crs(mun07_df.crs)


### Read in 2007 Area de Control information
adc07_df = gpd.read_file(adcshp)
adc07_df = adc07_df[~adc07_df['adcid'].isna()]
adc07_df = adc07_df[['adcid','tablaAlias','geometry']]
adc07_df.columns = ['adc07','adc07_type','geometry']
adc07_df.set_geometry('geometry', inplace=True)
adc07_df['muncode'] = adc07_df['adc07'].apply(lambda x: x[:5])

### Read in AMCA 2016 ADC info
adc16_df = gpd.read_file(ca2016adc)
adc16_df = adc16_df[['CONTROL','geometry']]
adc16_df.columns = ['adc16','geometry']
# adc16_df['cve_ent'] = adc16_df['adc16'].apply(lambda x: x[:2])
adc16_df.set_geometry('geometry', inplace=True)
adc16_df = adc16_df.set_crs(adc07_df.crs)
adc16_df['muncode'] = adc16_df['adc16'].apply(lambda x: x[:5])

In [13]:
### Compute agricultural land area for each ADC
agland_df = gpd.read_file(agland_path)
agland_df = agland_df[agland_df['type'] == 'AGRICOLA'].drop('type',axis=1)


In [24]:
ag_mun07_df = gpd.overlay(mun07_df, agland_df, how='intersection')
ag_mun07_df = ag_mun07_df.dissolve(by='muncode').reset_index()
ag_mun07_df.to_file(agland_07_mun, write_index=False)

ag_mun16_df = gpd.overlay(mun16_df, agland_df, how='intersection')
ag_mun16_df = ag_mun16_df.dissolve(by='muncode').reset_index()
ag_mun16_df.to_file(agland_16_mun, write_index=False)

In [46]:
adc07_inegi_agland_df, adc16_inegi_agland_df = gpd.GeoDataFrame(), gpd.GeoDataFrame()

for mun, subset in ag_mun16_df.groupby('muncode'):
    df = gpd.overlay(adc16_df[adc16_df['muncode'] == mun].drop('muncode',axis=1), subset, how='intersection')
    df['inegi_agland_area'] = df.to_crs(epsg=6372).geometry.area/10000.0
    adc16_inegi_agland_df = pd.concat([adc16_inegi_agland_df, df]).reset_index(drop=True)

adc16_inegi_agland_df.drop('muncode',axis=1).to_file(agland_16_adc,write_index=False)
    
for mun, subset in ag_mun07_df.groupby('muncode'):
    df = gpd.overlay(adc07_df[adc07_df['muncode'] == mun].drop('muncode',axis=1), subset, how='intersection')
    df['inegi_agland_area'] = df.to_crs(epsg=6372).geometry.area/10000.0
    adc07_inegi_agland_df = pd.concat([adc07_inegi_agland_df, df])

adc07_inegi_agland_df.drop('muncode',axis=1).to_file(agland_07_adc,write_index=False)
